# Import libraries

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("wandb")
!wandb login $secret_value

In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
pd.set_option('max_columns', None)
pd.set_option('max_rows', None)

import wandb
from wandb.keras import WandbCallback

from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.layers import Concatenate, LSTM, GRU, Masking, Add
from tensorflow.keras.layers import Bidirectional, Multiply
from tensorflow.keras import backend as K

# Set the random seeds
os.environ['TF_CUDNN_DETERMINISTIC'] = '1' 
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(42)
tf.random.set_seed(42)

# Custom function

In [ ]:
def plot_training(history):
    # list all data in history
    print(history.history.keys())
    # summarize history for loss
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'test'], loc='upper left')
    plt.show()

In [ ]:
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2    
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

# Load source datasets

In [ ]:
train_df = pd.read_csv('../input/ventilator-pressure-prediction/train.csv')
print(f"train_df: {train_df.shape}")
train_df.head()

In [ ]:
test_df = pd.read_csv('../input/ventilator-pressure-prediction/test.csv')
print(f"test_df: {test_df.shape}")
test_df.head()

# Feature Engineering

In [ ]:
from scipy.signal import hilbert, chirp
from scipy.signal import blackman
from scipy.fft import fft, fftfreq

N = 80
w = blackman(N+1)

ffta = lambda x: np.abs(fft(np.append(x.values,x.values[0]))[:80])
ffta.__name__ = 'ffta'

fftw = lambda x: np.abs(fft(np.append(x.values,x.values[0])*w)[:80])
fftw.__name__ = 'fftw'

In [ ]:
def add_features(df):
    df['cross']= df['u_in'] * df['u_out']
    df['cross2']= df['time_step'] * df['u_out']
    df['area'] = df['time_step'] * df['u_in']
    df['area'] = df.groupby('breath_id')['area'].cumsum()
    df['time_step_cumsum'] = df.groupby(['breath_id'])['time_step'].cumsum()
    df['u_in_cumsum'] = (df['u_in']).groupby(df['breath_id']).cumsum()
    print("Step-1...Completed")
    
    df['u_in_lag1'] = df.groupby('breath_id')['u_in'].shift(1)
    df['u_out_lag1'] = df.groupby('breath_id')['u_out'].shift(1)
    df['u_in_lag_back1'] = df.groupby('breath_id')['u_in'].shift(-1)
    df['u_out_lag_back1'] = df.groupby('breath_id')['u_out'].shift(-1)
    df['u_in_lag2'] = df.groupby('breath_id')['u_in'].shift(2)
    df['u_out_lag2'] = df.groupby('breath_id')['u_out'].shift(2)
    df['u_in_lag_back2'] = df.groupby('breath_id')['u_in'].shift(-2)
    df['u_out_lag_back2'] = df.groupby('breath_id')['u_out'].shift(-2)
    df['u_in_lag3'] = df.groupby('breath_id')['u_in'].shift(3)
    df['u_out_lag3'] = df.groupby('breath_id')['u_out'].shift(3)
    df['u_in_lag_back3'] = df.groupby('breath_id')['u_in'].shift(-3)
    df['u_out_lag_back3'] = df.groupby('breath_id')['u_out'].shift(-3)
    df['u_in_lag4'] = df.groupby('breath_id')['u_in'].shift(4)
    df['u_out_lag4'] = df.groupby('breath_id')['u_out'].shift(4)
    df['u_in_lag_back4'] = df.groupby('breath_id')['u_in'].shift(-4)
    df['u_out_lag_back4'] = df.groupby('breath_id')['u_out'].shift(-4)
    df = df.fillna(0)
    print("Step-2...Completed")
    
    df['breath_id__u_in__max'] = df.groupby(['breath_id'])['u_in'].transform('max')
    df['breath_id__u_in__mean'] = df.groupby(['breath_id'])['u_in'].transform('mean')
    df['breath_id__u_in__diffmax'] = df.groupby(['breath_id'])['u_in'].transform('max') - df['u_in']
    df['breath_id__u_in__diffmean'] = df.groupby(['breath_id'])['u_in'].transform('mean') - df['u_in']
    print("Step-3...Completed")
    
    df['u_in_diff1'] = df['u_in'] - df['u_in_lag1']
    df['u_out_diff1'] = df['u_out'] - df['u_out_lag1']
    df['u_in_diff2'] = df['u_in'] - df['u_in_lag2']
    df['u_out_diff2'] = df['u_out'] - df['u_out_lag2']
    df['u_in_diff3'] = df['u_in'] - df['u_in_lag3']
    df['u_out_diff3'] = df['u_out'] - df['u_out_lag3']
    df['u_in_diff4'] = df['u_in'] - df['u_in_lag4']
    df['u_out_diff4'] = df['u_out'] - df['u_out_lag4']
    print("Step-4...Completed")
    
    df['one'] = 1
    df['count'] = (df['one']).groupby(df['breath_id']).cumsum()
    df['u_in_cummean'] =df['u_in_cumsum'] /df['count']
    
    df['breath_id_lag']=df['breath_id'].shift(1).fillna(0)
    df['breath_id_lag2']=df['breath_id'].shift(2).fillna(0)
    df['breath_id_lagsame']=np.select([df['breath_id_lag']==df['breath_id']],[1],0)
    df['breath_id_lag2same']=np.select([df['breath_id_lag2']==df['breath_id']],[1],0)
    df['breath_id__u_in_lag'] = df['u_in'].shift(1).fillna(0)
    df['breath_id__u_in_lag'] = df['breath_id__u_in_lag'] * df['breath_id_lagsame']
    df['breath_id__u_in_lag2'] = df['u_in'].shift(2).fillna(0)
    df['breath_id__u_in_lag2'] = df['breath_id__u_in_lag2'] * df['breath_id_lag2same']
    print("Step-5...Completed")
    
    df['time_step_diff'] = df.groupby('breath_id')['time_step'].diff().fillna(0)
#     df['ewm_u_in_mean'] = (df\
#                            .groupby('breath_id')['u_in']\
#                            .ewm(halflife=9)\
#                            .mean()\
#                            .reset_index(level=0,drop=True))
    df[["15_in_sum","15_in_min","15_in_max","15_in_mean"]] = (df\
                                                              .groupby('breath_id')['u_in']\
                                                              .rolling(window=15,min_periods=1)\
                                                              .agg({"15_in_sum":"sum",
                                                                    "15_in_min":"min",
                                                                    "15_in_max":"max",
                                                                    "15_in_mean":"mean"})\
                                                               .reset_index(level=0,drop=True))
    print("Step-6...Completed")
    
    df['u_in_lagback_diff1'] = df['u_in'] - df['u_in_lag_back1']
    df['u_out_lagback_diff1'] = df['u_out'] - df['u_out_lag_back1']
    df['u_in_lagback_diff2'] = df['u_in'] - df['u_in_lag_back2']
    df['u_out_lagback_diff2'] = df['u_out'] - df['u_out_lag_back2']
    print("Step-7...Completed")
    
    df['R'] = df['R'].astype(str)
    df['C'] = df['C'].astype(str)
    df['R__C'] = df["R"].astype(str) + '__' + df["C"].astype(str)
    df = pd.get_dummies(df)
    print("Step-8...Completed")
    
    # https://www.kaggle.com/lucasmorin/spectral-analysis-feature-engineering#Feature-importance
    df['fft_u_in'] = df.groupby('breath_id')['u_in'].transform(ffta)
    df['fft_u_in_w'] = df.groupby('breath_id')['u_in'].transform(fftw)
    df['analytical'] = df.groupby('breath_id')['u_in'].transform(hilbert)
    df['envelope'] = np.abs(df['analytical'])
    df['phase'] = np.angle(df['analytical'])
    df['unwrapped_phase'] = df.groupby('breath_id')['phase'].transform(np.unwrap)
    df['phase_shift1'] = df.groupby('breath_id')['unwrapped_phase'].shift(1).astype(np.float32)
    df['IF'] = df['unwrapped_phase'] - df['phase_shift1'].astype(np.float32)
    df = df.fillna(0)
    df = df.drop('analytical',axis=1)
    
    # https://www.kaggle.com/c/ventilator-pressure-prediction/discussion/280471
    df['time_gap'] = df['time_step'] - df['time_step'].shift(1).fillna(0)
    df['u_in_gap'] = df['u_in'] - df['u_in'].shift(1).fillna(0)
    df['u_in_rate'] = df['u_in_gap'] / df['time_gap']

    df.loc[list(range(0, len(df), 80)), 'time_gap'] = 0
    df.loc[list(range(0, len(df), 80)), 'u_in_gap'] = 0
    df.loc[list(range(0, len(df), 80)), 'u_in_rate'] = 0
    
    # Forgot source: https://www.kaggle.com/illidan7/gbvpp-feat-eng
    g = df.groupby('breath_id')['u_in']
    df['ewm_u_in_mean'] = g.ewm(halflife=10).mean()\
                           .reset_index(level=0, drop=True)
    df['ewm_u_in_std'] = g.ewm(halflife=10).std()\
                          .reset_index(level=0, drop=True)
    df['ewm_u_in_corr'] = g.ewm(halflife=10).corr()\
                           .reset_index(level=0, drop=True)
    
    df['expand_mean'] = g.expanding(2).mean()\
                         .reset_index(level=0, drop=True)
    df['expand_max'] = g.expanding(2).max()\
                        .reset_index(level=0, drop=True)
    df['expand_std'] = g.expanding(2).std()\
                        .reset_index(level=0, drop=True)
    
    df = df.fillna(0)
    
    print("Step-9...Completed")
       
    
    return df


print("Train data...\n")
train = add_features(train_df)

print("\nTest data...\n")
test = add_features(test_df)

del train_df
del test_df
gc.collect()

print(train.shape)
print(test.shape)

# Drop columns

In [ ]:
targets = train[['pressure']].to_numpy().reshape(-1, 80)

train.drop(['pressure','id', 'breath_id','one','count',
            'breath_id_lag','breath_id_lag2','breath_id_lagsame',
            'breath_id_lag2same'], axis=1, inplace=True)

test = test.drop(['id', 'breath_id','one','count','breath_id_lag',
                  'breath_id_lag2','breath_id_lagsame',
                  'breath_id_lag2same'], axis=1)

print(f"train: {train.shape} \ntest: {test.shape}")

In [ ]:
dropCols = ['u_in_lag_back1','u_out_lag_back1','u_in_lag2','u_out_lag2','u_in_lag_back2','u_out_lag_back2',
'u_in_lag3','u_out_lag3','u_in_lag_back3','u_out_lag_back3','u_in_lag4','u_out_lag4','u_in_lag_back4',
'u_out_lag_back4','breath_id__u_in__max','breath_id__u_in__mean']

train.drop(columns=dropCols, inplace=True)
test.drop(columns=dropCols, inplace=True)

print(f"train: {train.shape} \ntest: {test.shape}")

In [ ]:
# Drop columns based on permutation importance
dropCols = ['breath_id__u_in__diffmax','ewm_u_in_corr','u_in_grp_R__C_steps_max_diff','u_out_diff1',
'u_out_lagback_diff2','u_out_lagback_diff1','u_out_lag3','u_out_lag2','unwrapped_phase',
'u_in_grp_R__C_steps_max','phase_shift1','u_in_grp_R__C_steps_min','u_out_lag1','u_out_lag4',
'u_out_lag_back2','u_out_lag_back1','u_out_lag_back4','u_out_lag_back3','cross2',
'time_step_cumsum','plus_one','minus_one','time_step','u_in_first','breath_id__u_in__diffmean',
'ewm_u_in_std','phase','area','expand_max','breath_id__u_in__max','expand_std']
dropCols = [col for col in dropCols if col in train.columns]

train.drop(columns=dropCols, inplace=True)
test.drop(columns=dropCols, inplace=True)

COLS = train.columns.tolist()

print(train.shape)
print(test.shape)

In [ ]:
# Drop columns based on permutation importance
dropCols = ['time_passed','time_passed_grp_R__C_steps_mean','time_passed_grp_R__C_steps_min','u_out_diff2','u_out_diff4',
'u_in_rolling_mean_lead15','u_in_rolling_max_lead15','u_in_rolling_min_lead15','u_in_rolling_std_lead15',
'fft_u_in','fft_u_in_w','envelope','IF','time_gap']
dropCols = [col for col in dropCols if col in train.columns]

train.drop(columns=dropCols, inplace=True)
test.drop(columns=dropCols, inplace=True)

COLS = train.columns.tolist()

print(train.shape)
print(test.shape)

In [ ]:
train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

# Prepare data for modeling

In [ ]:
# quantVal = 15

# scaler = RobustScaler(quantile_range=(quantVal, 100-quantVal))
scaler = RobustScaler()
train = scaler.fit_transform(train)
test = scaler.transform(test)

train = train.reshape(-1, 80, train.shape[-1])
test = test.reshape(-1, 80, train.shape[-1])
print(f"train: {train.shape} \n")
print(f"test: {test.shape} \n")
print(f"\n\ntargets: {targets.shape}")

In [ ]:
pressure = targets.squeeze().reshape(-1,1).astype('float32')

P_MIN = np.min(pressure)
P_MAX = np.max(pressure)
P_STEP = (pressure[1] - pressure[0])[0]
print('Min pressure: {}'.format(P_MIN))
print('Max pressure: {}'.format(P_MAX))
print('Pressure step: {}'.format(P_STEP))
print('Unique values:  {}'.format(np.unique(pressure).shape[0]))

del pressure
_ = gc.collect()

# Hardware config

In [ ]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
    BATCH_SIZE = strategy.num_replicas_in_sync * 64
    print("Running on TPU:", tpu.master())
    print(f"Batch Size: {BATCH_SIZE}")
    
except ValueError:
    strategy = tf.distribute.get_strategy()
    BATCH_SIZE = 512
    print(f"Running on {strategy.num_replicas_in_sync} replicas")
    print(f"Batch Size: {BATCH_SIZE}")

# IlliDub Model

In [ ]:
def IlliDubModel():
    
    x_input = Input(shape=(train.shape[-2:]))
    
    x1 = Bidirectional(LSTM(units=768, return_sequences=True))(x_input)
    x2 = Bidirectional(LSTM(units=512, return_sequences=True))(x1)
    x3 = Bidirectional(LSTM(units=256, return_sequences=True))(x2)
    
    z2 = Bidirectional(GRU(units=256, return_sequences=True))(x2)
    z3 = Bidirectional(GRU(units=128, return_sequences=True))(Add()([x3, z2]))
    
    x = Concatenate(axis=2)([x3, z2, z3])
    x = Bidirectional(LSTM(units=192, return_sequences=True))(x)
    
    x = Dense(units=128, activation='selu')(x)
    
    x_output = Dense(units=1)(x)

    model = Model(inputs=x_input, outputs=x_output, 
                  name='DNN_Model')
    return model

In [ ]:
model = IlliDubModel()
model.summary()

In [ ]:
plot_model(
    model, 
    to_file='IlliDub_model.png', 
    show_shapes=True,
    show_layer_names=True
)

# Model training

In [ ]:
VERBOSE = 1
ONE_FOLD_ONLY = False 
COMPUTE_LSTM_IMPORTANCE = False

FOLDS = 7
SEED = 101

# Learning rate
INIT_LR = 0.001

# ReduceLROnPlateau
FACTOR = 0.8
LR_PATIENCE = 4
MIN_LR = 5e-6
LR_DELTA = 0

# Early Stopping
ES_DELTA = 0
PATIENCE = 25

# Fit
EPOCHS = 300

In [ ]:
with strategy.scope():

    test_preds = []
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

    for fold, (train_idx, test_idx) in enumerate(kf.split(train, targets)):


        print(f"Fold {fold+1}")
        X_train, X_valid = train[train_idx], train[test_idx]
        y_train, y_valid = targets[train_idx], targets[test_idx]

        if not ONE_FOLD_ONLY:

            print("Init wandb logs")

            # Initialize wandb with your project name
            run = wandb.init(project='GBVPP',
                             config={  # and include hyperparameters and metadata
                                 "FOLDS": FOLDS,
                                 "SEED": SEED,
#                                  "ROBUSTSCALER QUANT": quantVal,
                                 "INIT_LR": INIT_LR,
                                 "RLROP FACTOR": FACTOR,
                                 "RLROP LR_PATIENCE": LR_PATIENCE,
                                 "RLROP MIN_LR": MIN_LR,
                                 "RLROP DELTA": LR_DELTA,
                                 "ES DELTA": ES_DELTA,
                                 "PATIENCE": PATIENCE,
                                 "EPOCHS": EPOCHS,
                                 "BATCH_SIZE": BATCH_SIZE,
                                 "LOSS": "mae",
                                 "MODEL": "IlliDubModel", 

                             })
            config = wandb.config

            # Wandb
            wnb = WandbCallback()

        print("Model init")

        model = IlliDubModel()
        model.compile(optimizer="adam", loss="mae") #, sample_weight_mode="temporal"

        # ModelCheckpoint
        save_locally = tf.saved_model.SaveOptions(experimental_io_device='/job:localhost')
        chk_point = ModelCheckpoint(f'./IlliDubModel_{SEED}_fold{fold+1}.h5', options=save_locally, 
                                    monitor='val_loss', verbose=VERBOSE, 
                                    save_best_only=True, mode='min')

        # Learning Rate Schedule
        lr = ReduceLROnPlateau(monitor="val_loss", factor=FACTOR, 
                               patience=LR_PATIENCE, verbose=VERBOSE,
                               min_lr=MIN_LR, min_delta=LR_DELTA)

        # Early Stopping Criterion
        es = EarlyStopping(monitor="val_loss", patience=PATIENCE, 
                           verbose=VERBOSE, mode="min", 
                           restore_best_weights=True,
                           min_delta=ES_DELTA)



        # Set Learning Rate
        K.set_value(model.optimizer.learning_rate, INIT_LR)
        print("Initial Learning rate:", model.optimizer.learning_rate.numpy())

        # Callbacks
        if ONE_FOLD_ONLY:
            callbacks = [lr, chk_point, es]
        else:
            callbacks = [lr, chk_point, es, wnb]

        # Model training
        print("Model training begins")
        history = model.fit(X_train, y_train, 
                          validation_data=(X_valid, y_valid), 
                          epochs=EPOCHS,
                          verbose=VERBOSE,
                          batch_size=BATCH_SIZE, 
                          callbacks=callbacks)

        plot_training(history)

        if COMPUTE_LSTM_IMPORTANCE:
            results = []
            print(' Computing LSTM feature importance...')

            # COMPUTE BASELINE (NO SHUFFLE)
            oof_preds = model.predict(X_valid, verbose=0).squeeze() 
            baseline_mae = np.mean(np.abs( oof_preds-y_valid ))
            results.append({'feature':'BASELINE','mae':baseline_mae})           

            for k in tqdm(range(len(COLS))):

                # SHUFFLE FEATURE K
                save_col = X_valid[:,:,k].copy()
                np.random.shuffle(X_valid[:,:,k])

                # COMPUTE OOF MAE WITH FEATURE K SHUFFLED
                oof_preds = model.predict(X_valid, verbose=0).squeeze() 
                mae = np.mean(np.abs( oof_preds-y_valid ))
                results.append({'feature':COLS[k],'mae':mae})
                X_valid[:,:,k] = save_col

            # DISPLAY LSTM FEATURE IMPORTANCE
            print()
            df = pd.DataFrame(results)
            df = df.sort_values('mae')
            plt.figure(figsize=(10,20))
            plt.barh(np.arange(len(COLS)+1),df.mae)
            plt.yticks(np.arange(len(COLS)+1),df.feature.values)
            plt.title('LSTM Feature Importance',size=16)
            plt.ylim((-1,len(COLS)+1))
            plt.plot([baseline_mae,baseline_mae],[-1,len(COLS)+1], '--', color='orange',
                     label=f'Baseline OOF\nMAE={baseline_mae:.3f}')
            plt.xlabel(f'Fold {fold+1} OOF MAE with feature permuted',size=14)
            plt.ylabel('Feature',size=14)
            plt.legend()
            plt.show()

            # SAVE LSTM FEATURE IMPORTANCE
            df = df.sort_values('mae',ascending=False)
            df.to_csv(f'lstm_feature_importance_fold_{fold+1}.csv',index=False)

        # ONLY DO ONE FOLD
        if ONE_FOLD_ONLY: break

        ######################################################



        load_locally = tf.saved_model.LoadOptions(experimental_io_device='/job:localhost')
        model = load_model(f'./IlliDubModel_{SEED}_fold{fold+1}.h5', options=load_locally)

        print("Calculate OOF score")
        y_true = y_valid.squeeze().reshape(-1, 1)
        y_pred = model.predict(X_valid, batch_size=BATCH_SIZE).squeeze().reshape(-1, 1)
        score = mean_absolute_error(y_true, y_pred)
        print(f"Fold-{fold+1} | OOF Score: {score}")

        test_preds.append(model.predict(test, batch_size=BATCH_SIZE).squeeze().reshape(-1, 1).squeeze())


    if not COMPUTE_LSTM_IMPORTANCE: wandb.finish()

# Create submission file

In [ ]:
for i in (range(FOLDS)):
    submission = pd.read_csv('../input/ventilator-pressure-prediction/sample_submission.csv')
    submission["pressure"] = test_preds[i]
    submission.to_csv(f'IlliDubModel_fold{i+1}_submission.csv', index=False)

In [ ]:
submission = pd.read_csv('../input/ventilator-pressure-prediction/sample_submission.csv')
submission["pressure"] = sum(test_preds)/FOLDS
submission["pressure"] = np.round((submission.pressure - P_MIN)/P_STEP) * P_STEP + P_MIN
submission["pressure"] = np.clip(submission.pressure, P_MIN, P_MAX)
submission.to_csv('mean_submission.csv', index=False)

In [ ]:
submission = pd.read_csv('../input/ventilator-pressure-prediction/sample_submission.csv')
submission["pressure"] = np.median(np.vstack(test_preds),axis=0)
submission["pressure"] = np.round((submission.pressure - P_MIN)/P_STEP) * P_STEP + P_MIN
submission["pressure"] = np.clip(submission.pressure, P_MIN, P_MAX)
submission.to_csv('median_submission.csv', index=False)